In [ ]:
# M3: DEGRADATION TRAJECTORY STAGER (3-Class XGBoost)
# Classes: Healthy (stage 0-1), Warning (stage 2), Critical (stage 3-4)
# Top 1% features: ordinal weights, monotonic enforcement, transition probability

import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix

session = get_active_session()
session.sql("USE DATABASE FAILURE_GENOME_DB").collect()
session.sql("USE SCHEMA ML_MODELS").collect()
print("Session ready.")

In [ ]:
df = session.table("FAILURE_GENOME_DB.ML_FEATURES.TRAIN_TRAJECTORY_3CLASS").to_pandas()
print(f"Loaded {len(df)} rows")
print(f"\n3-Class distribution:")
print(df['STAGE_3CLASS'].value_counts().sort_index().rename({0:'Healthy', 1:'Warning', 2:'Critical'}))

In [ ]:
exclude_cols = ['ASSET_ID', 'TIMESTAMP', 'FAILURE_MODE', 'ORIGINAL_STAGE', 'STAGE_3CLASS']
feature_cols = [c for c in df.columns if c not in exclude_cols]
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

df = df.sort_values('TIMESTAMP').reset_index(drop=True)
split_idx = int(len(df) * 0.8)
X_train, X_test = df.iloc[:split_idx][feature_cols].values, df.iloc[split_idx:][feature_cols].values
y_train, y_test = df.iloc[:split_idx]['STAGE_3CLASS'].values, df.iloc[split_idx:]['STAGE_3CLASS'].values

# Aggressive ordinal weights
stage_weights = {0: 0.5, 1: 5.0, 2: 2.0}
sample_weights = np.array([stage_weights[s] for s in y_train])
print(f"Train: {len(X_train)} | Test: {len(X_test)} | Features: {len(feature_cols)}")

In [ ]:
model = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.08,
    objective='multi:softprob', num_class=3,
    reg_alpha=0.1, reg_lambda=1.0, min_child_weight=3,
    random_state=42, n_jobs=-1
)
model.fit(X_train, y_train, sample_weight=sample_weights, 
          eval_set=[(X_test, y_test)], verbose=False)
print("Training complete.")

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

f1 = f1_score(y_test, y_pred, average='macro')
mae = np.mean(np.abs(y_test - y_pred))
within_one = np.mean(np.abs(y_test - y_pred) <= 1)

print(f"F1 Macro: {f1:.4f}")
print(f"Mean Absolute Stage Error: {mae:.3f}")
print(f"Within ±1 Accuracy: {within_one:.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Healthy','Warning','Critical'])}")
print("Confusion Matrix:")
print(pd.DataFrame(confusion_matrix(y_test, y_pred), 
    index=['Actual_H','Actual_W','Actual_C'], columns=['Pred_H','Pred_W','Pred_C']))

In [ ]:
importance = pd.DataFrame({
    'feature': feature_cols, 'importance': model.feature_importances_
}).sort_values('importance', ascending=False).head(15)
print("Top 15 Features:")
print(importance.to_string(index=False))

In [ ]:
import os
from snowflake.ml.registry import Registry
from snowflake.ml.model import custom_model

model_path = "/tmp/stager_3class_v2.json"
model.save_model(model_path)

class DegradationStager3Class(custom_model.CustomModel):
    def __init__(self, context):
        super().__init__(context)
        
    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        import numpy as np
        import xgboost as xgb
        
        xgb_model = xgb.XGBClassifier()
        xgb_model.load_model(self.context.path("xgb_model"))
        
        exclude = {'ASSET_ID','TIMESTAMP','FAILURE_MODE','DEGRADATION_STAGE','ORIGINAL_STAGE','STAGE_3CLASS'}
        cols = [c for c in input_df.columns if c not in exclude]
        X = input_df[cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
        
        preds = xgb_model.predict(X)
        proba = xgb_model.predict_proba(X)
        labels = ['Healthy','Warning','Critical']
        
        transition = []
        for i in range(len(preds)):
            s = int(preds[i])
            transition.append(0.0 if s >= 2 else float(proba[i, s+1:].sum()))
        
        return pd.DataFrame({
            'PREDICTED_STAGE': [labels[p] for p in preds],
            'STAGE_CONFIDENCE': proba.max(axis=1).round(4),
            'TRANSITION_PROBABILITY': np.round(transition, 4),
            'PROB_HEALTHY': proba[:, 0].round(4),
            'PROB_WARNING': proba[:, 1].round(4),
            'PROB_CRITICAL': proba[:, 2].round(4)
        })

reg = Registry(session=session, database_name="FAILURE_GENOME_DB", schema_name="ML_MODELS")
try:
    reg.delete_model("DEGRADATION_STAGER")
except:
    pass

stager = DegradationStager3Class(
    context=custom_model.ModelContext(artifacts={"xgb_model": model_path})
)
sample_input = pd.DataFrame(np.zeros((1, len(feature_cols))), columns=feature_cols)

mv = reg.log_model(
    stager, model_name="DEGRADATION_STAGER", version_name="V2",
    metrics={"f1_macro": float(f1), "mae": float(mae), "within_one_acc": float(within_one)},
    sample_input_data=sample_input,
    target_platforms=["WAREHOUSE"],
    comment=f"XGBoost 3-class stager. F1={f1:.4f}"
)
print(f"Registered: DEGRADATION_STAGER V2 | F1: {f1:.4f}")

In [ ]:
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

print("Top 15 Features for Degradation Staging:")
print(importance.to_string(index=False))

In [ ]:
import os
from snowflake.ml.registry import Registry
from snowflake.ml.model import custom_model

model_path = "/tmp/stager_3class_v2.json"
model.save_model(model_path)
print(f"Model saved to {model_path}")

class DegradationStager3Class(custom_model.CustomModel):
    def __init__(self, context):
        super().__init__(context)
        
    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        import numpy as np
        import xgboost as xgb
        
        xgb_model = xgb.XGBClassifier()
        xgb_model.load_model(self.context.path("xgb_model"))
        
        exclude = {'ASSET_ID','TIMESTAMP','FAILURE_MODE','DEGRADATION_STAGE','ORIGINAL_STAGE','STAGE_3CLASS'}
        cols = [c for c in input_df.columns if c not in exclude]
        X = input_df[cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
        
        preds = xgb_model.predict(X)
        proba = xgb_model.predict_proba(X)
        labels = ['Healthy','Warning','Critical']
        
        transition = []
        for i in range(len(preds)):
            s = int(preds[i])
            transition.append(0.0 if s >= 2 else float(proba[i, s+1:].sum()))
        
        return pd.DataFrame({
            'PREDICTED_STAGE': [labels[p] for p in preds],
            'STAGE_CONFIDENCE': proba.max(axis=1).round(4),
            'TRANSITION_PROBABILITY': np.round(transition, 4),
            'PROB_HEALTHY': proba[:, 0].round(4),
            'PROB_WARNING': proba[:, 1].round(4),
            'PROB_CRITICAL': proba[:, 2].round(4)
        })

reg = Registry(session=session, database_name="FAILURE_GENOME_DB", schema_name="ML_MODELS")
try:
    reg.delete_model("DEGRADATION_STAGER")
except:
    pass

stager = DegradationStager3Class(
    context=custom_model.ModelContext(artifacts={"xgb_model": model_path})
)
sample_input = pd.DataFrame(np.zeros((1, len(feature_cols))), columns=feature_cols)

mv = reg.log_model(
    stager, model_name="DEGRADATION_STAGER", version_name="V2",
    metrics={"f1_macro": float(f1), "mae": float(mae), "within_one_acc": float(within_one)},
    sample_input_data=sample_input,
    target_platforms=["WAREHOUSE"],
    comment=f"XGBoost 3-class stager (Healthy/Warning/Critical). F1={f1:.4f}"
)
print(f"Registered: DEGRADATION_STAGER V2 | F1: {f1:.4f} | MAE: {mae:.3f} | ±1 Acc: {within_one:.4f}")

In [ ]:
# Inference test from registry
reg_model = reg.get_model("DEGRADATION_STAGER").version("V2")
test_sample = pd.DataFrame(X_test[:5], columns=feature_cols)
result = reg_model.run(test_sample, function_name="predict")
print("Inference test (3-class from registry):")
print(result)
print(f"\nActual: {['Healthy' if s==0 else 'Warning' if s==1 else 'Critical' for s in y_test[:5]]}")
print(f"\nModel metrics: F1={f1:.4f} | MAE={mae:.3f} | ±1 Acc={within_one:.4f}")